In [1]:
from __future__ import annotations 

import random
from typing import Optional

In [2]:
import carla

In [3]:
CARLA_HOST = "localhost"
CARLA_PORT = 2000
CARLA_TIMEOUT_SECONDS = 10.0
RANDOM_SEED = 42

In [4]:
def connect_to_carla() -> tuple[carla.Client, carla.World]:
    '''Create a CARLA client and return the connected world.'''

    client = carla.Client(CARLA_HOST, CARLA_PORT)
    client.set_timeout(CARLA_TIMEOUT_SECONDS)

    world = client.get_world()

    return client, world

In [5]:
def get_pedestrian_blueprint(blueprint_library: carla.BlueprintLibrary) -> carla.ActorBlueprint:
    '''Return a randomly selected pedestrian blueprint.'''

    pedestrian_blueprint = blueprint_library.filter("walker.pedestrian.*")

    if not pedestrian_blueprint:
        raise RuntimeError("No pedestrian blueprint found.")

    return random.choice(pedestrian_blueprint)


In [6]:
def get_spawn_location(world: carla.World) -> carla.Transform:
    '''Generate a pedestrian spawn location using the navigation mesh.'''

    location = world.get_random_location_from_navigation()

    if location is None:
        raise RuntimeError(
            "CARLA could not find a valid pedestrian navigation location"
        )

    return carla.Transform(location)

In [7]:
def spawn_pedestrian(world: carla.World, blueprint_library: carla.BlueprintLibrary) -> carla.Walker:
    '''Spawn a single pedestrian actor'''

    pedestrian_blueprint = get_pedestrian_blueprint(blueprint_library)
    spawn_transform = get_spawn_location(world)

    pedestrian = world.try_spawn_actor(
        pedestrian_blueprint,
        spawn_transform,
    )

    if pedestrian is None:
        raise RuntimeError(
            "Failed to spawn pedestrian."
        )

    return pedestrian


In [8]:
def main() -> None:
    '''Spawn and report a single pedestrian.'''

    random.seed(RANDOM_SEED)

    client, world = connect_to_carla()

    blueprint_library = world.get_blueprint_library()

    pedestrian = spawn_pedestrian(
        world,
        blueprint_library,
    )

    location = pedestrian.get_location()

    print("=== PEDESTRIAN SPAWNED ===")
    print(f"Actor ID: {pedestrian.id}")
    print(f"Type: {pedestrian.type_id}")
    print(
        f"Location"
        f"x={location.x:.2f}, "
        f"y={location.y:.2f}, "
        f"z={location.z:.2f}"
    )

In [9]:
if __name__ == "__main__":
    main()

=== PEDESTRIAN SPAWNED ===
Actor ID: 24
Type: walker.pedestrian.0035
Locationx=0.00, y=0.00, z=0.00
